# 05 · Sister-pair study

The study-design pipeline (`pipeline.py`): story cards, PFAS exposure, LLM pairwise judgments, and Bradley-Terry / Davidson ranking with the sister-pair test.

Run the steps from the project root first (see README). This notebook only reads their outputs, and the power analysis below needs no data or API key.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable when running from notebooks/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

from src import config
from src.data_loader import load_participants
from src.simulate import power

participants = pd.DataFrame([p.__dict__ for p in load_participants()])
participants

## Power analysis

How likely is the sister-pair test to detect an effect (in standard deviations of individual risk) for a given number of sister pairs? Each point is 100 simulated studies.

In [ ]:
effects = [0.25, 0.5, 0.75, 1.0]
grid = {n: [power(n, e, n_sims=100) for e in effects] for n in (10, 20, 30, 40)}

for n, row in grid.items():
    plt.plot(effects, row, marker="o", label=f"{n} sister pairs")
plt.axhline(0.8, color="grey", linestyle="--", linewidth=1)
plt.xlabel("effect size (SD)")
plt.ylabel("power (alpha = 0.05)")
plt.legend()
plt.title("Sister-pair test power")
plt.show()

## PFAS exposure

From `python pipeline.py exposure` (requires the licensed tracker file).

In [ ]:
if config.EXPOSURE_FILE.exists():
    exposure = json.loads(config.EXPOSURE_FILE.read_text())
    display(pd.DataFrame({pid: e["summary"] for pid, e in exposure.items()}).T)
else:
    print("Run: python pipeline.py exposure")

## Ranking and sister-pair test

From `python pipeline.py rank`.

In [ ]:
if config.RANKING_FILE.exists():
    report = json.loads(config.RANKING_FILE.read_text())
    print("Judge order-swap consistency:", report["judge_order_consistency"]["overall"])
    rows = []
    for key, r in report["results"].items():
        row = {"domain": key, "nu": r["davidson"]["nu"], "comparisons": r["n_comparisons"]}
        row.update({k: v for k, v in r.get("sister_test", {}).items() if k in ("statistic", "p_value")})
        rows.append(row)
    display(pd.DataFrame(rows).set_index("domain"))
else:
    print("Run: python pipeline.py compare, then python pipeline.py rank")